# Lesson 12 — Adversarial evasion, and how metrics lie

Lesson 12 — Adversarial evasion, and how metrics lie
====================================================

GURU NOTE
---------
You can train a model. Now learn to DISTRUST it the way a security engineer must.
Two hard truths that separate a practitioner from a tutorial-follower:

  A. ATTACKERS ADAPT. A payload can be rewritten to mean the same thing but LOOK
     different, dodging a model that only saw the original form. Your repo has a
     whole module for this (mutate.py: case-swap, /**/ comments, $IFS, ca$1t...)
     mirroring the WAF-A-MoLE evasion research. We'll evade a model, then note the
     fix: train on the mutations too.

  B. YOUR METRICS PROBABLY LIE. If near-duplicate rows land in BOTH the training
     and validation split, the model is graded partly on things it memorized, and
     your AUC looks amazing while real-world performance is worse. This is the #1
     criticism of the xorguard-ml eval today: generate.py/hardneg.py manufacture
     correlated rows, and train.split() shuffles by ROW, so copies leak across the
     split. We'll MEASURE the inflation, then show the fix (split by GROUP).

This lesson is the difference between "my model scores 0.99" and "my model scores
0.99 and I know why that number is trustworthy."

Run me:  python lessons/lesson_12_adversarial_and_pitfalls.py

## Setup

These notebooks *import* the real code from `lessons/*.py` — one source of truth, no forked copies. This cell finds that folder.

In [1]:
import sys, pathlib
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / 'lessons' / '_util.py').exists():
        sys.path.insert(0, str(base / 'lessons'))
        print('lessons on path:', base / 'lessons'); break
else:
    raise RuntimeError('Open this notebook inside the xorguard-ml-course folder.')

lessons on path: /Users/anil/Documents/developer/vscodeworkspaces/xordev/xorguard-ml-course/lessons


## The key code (shown live from the source file)

Read it, then copy any function into a new cell and tweak it — the originals live in `lessons/lesson_12_adversarial_and_pitfalls.py`.

In [2]:
import inspect
from lesson_12_adversarial_and_pitfalls import case_swap, comment_inject, ifs_sub
for fn in (case_swap, comment_inject, ifs_sub):
    print(inspect.getsource(fn)); print('-' * 72)

def case_swap(s: str) -> str:
    return "".join(c.upper() if i % 2 == 0 else c.lower() for i, c in enumerate(s))

------------------------------------------------------------------------
def comment_inject(s: str) -> str:
    # SQL treats /**/ as whitespace; the query still runs.
    return "/**/".join(s.split(" "))

------------------------------------------------------------------------
def ifs_sub(s: str) -> str:
    # the shell treats $IFS as a field separator == space.
    return s.replace(" ", "${IFS}")

------------------------------------------------------------------------


## Run the full narrated lesson

This executes the same walkthrough as `python lessons/lesson_12_adversarial_and_pitfalls.py`, printing each step with its result.

In [3]:
from lesson_12_adversarial_and_pitfalls import main
main()


════════════════════════════════════════════════════
Lesson 12 — Adversarial evasion, and how metrics lie
════════════════════════════════════════════════════

▸ PART A — train a text MLP, then EVADE it with meaning-preserving rewrites.
    Each row: the model's attack-score for the SAME attack, rewritten.
    payload                                           original    case-swap  /**/comment         $IFS
    q=' OR 1=1--                                         0.999        0.070        0.149        0.308
    id=5' UNION SELECT password FROM users--             0.996        0.854        0.986        0.962
    q=; cat /etc/passwd                                  0.998        0.966        0.942        0.888

    3/9 rewrites pushed a caught attack BELOW 0.5 — evaded.
    The model never saw these forms, so its n-gram vector shifted enough
    to slip past. An attacker with mutate.py can hunt for exactly these.

▸ The fix: ADVERSARIAL TRAINING — feed the mutations in as attacks too.


    After adversarial training: 9/9 of those same
    rewrites are caught again. This is why mutate.py exists in your repo,
    and why the engine has a CI 'mutation exam' that must stay >=85%.

▸ PART B — how duplicate rows across a split INFLATE your metrics.
    Setup: 80 distinct base payloads, each DUPLICATED 15x with a throwaway
    suffix — the way hardneg.py repeats each row 41x and wrap_payload emits
    3 rows per payload. To isolate MEMORIZATION from real learning, each
    base's label is a per-payload property with no transferable pattern
    (its distinctive token is the only clue). So any success on UNSEEN
    bases must be luck, and any success on SEEN bases is pure memorization.

    split-by-ROW   (leaky):  validation accuracy = 100.0%
    split-by-GROUP (honest): validation accuracy = 56.7%  (~ chance)
    INFLATION from leakage: +43.3%
    Same data, same model. Row-splitting let the model find each val row's
    duplicate in training and copy the answer, so it look

## Exercises

Try these in new cells below. Worked answers are in `solutions/lesson_12_solution.py`.

```
1. Add a new mutation (e.g. URL-encode the payload, or insert a null byte) and
   see if it evades the adversarially-trained model. The arms race never ends;
   which transformations generalize and which don't?

2. In Part B, raise the per-base repeat count from 15 to 60. Does the row-split
   inflation get WORSE? Explain why more duplication widens the leakage gap.

3. Reproduce the leakage on the REAL repo's shape: write a group-aware split for
   lesson_11's generate_corpus (group by family+template) and compare the
   capstone's detect-recall under row-split vs group-split. Report both honestly.
```

In [4]:
# your work here
